In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
class SalesDataAnalyzer:
    def __init__(self, file_path="sales_data.csv"):
        self.data = pd.DataFrame()
        self.file_path = file_path
        self.last_plot = None
        self.load_data(file_path)
    def __del__(self):
        pass
    def load_data(self, file_path):
        try:
            if not os.path.isfile(file_path):
                print(f"File not found: {file_path}")
                print("Place sales_data.csv beside this script, or use menu option 1.")
                return False
            data = pd.read_csv(file_path)
            if data.empty:
                print("Warning: The dataset is empty.")
            if "Date" in data.columns:
                data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
            for col in ["Sales", "Profit", "Quantity"]:
                if col in data.columns:
                    data[col] = pd.to_numeric(data[col], errors="coerce")
            self.data = data
            self.file_path = file_path
            print("Dataset loaded successfully!")
            print(f"Rows: {len(self.data)} | Columns: {len(self.data.columns)}")
            return True
        except (OSError, pd.errors.ParserError, ValueError) as error:
            print("Error loading dataset:", error)
            return False
    def check_data(self):
        if self.data.empty:
            print("Please load a valid dataset first.")
            return False
        return True
    def select_column(self, numeric_only=False):
        columns = (
            self.data.select_dtypes(include=np.number).columns.tolist()
            if numeric_only else self.data.columns.tolist()
        )
        if not columns:
            print("No suitable columns available.")
            return None
        print("Available columns:", ", ".join(columns))
        column = input("Enter column name: ").strip()
        if column not in columns:
            print("Invalid column name!")
            return None
        return column
    def explore_data(self):
        if not self.check_data():
            return
        print("\n===== EXPLORE DATA =====")
        print("1. First 5 rows\n2. Last 5 rows\n3. Column names")
        print("4. Data types\n5. Dataset information\n6. Dataset shape")
        try:
            choice = int(input("Enter choice: "))
            if choice == 1:
                print(self.data.head())
            elif choice == 2:
                print(self.data.tail())
            elif choice == 3:
                print(self.data.columns.tolist())
            elif choice == 4:
                print(self.data.dtypes)
            elif choice == 5:
                self.data.info()
            elif choice == 6:
                print("Rows, Columns:", self.data.shape)
            else:
                print("Invalid choice!")
        except ValueError:
            print("Enter a valid menu number.")
    def clean_data(self):
        if not self.check_data():
            return
        print("\n===== CLEAN DATA =====")
        print("1. Display rows with missing values")
        print("2. Fill numeric missing values with mean")
        print("3. Fill missing values with a specific value")
        print("4. Drop rows with missing values")
        print("5. Remove duplicate rows")
        try:
            choice = int(input("Enter choice: "))
            if choice == 1:
                missing = self.data[self.data.isnull().any(axis=1)]
                print(missing if not missing.empty else "No missing values found.")
            elif choice == 2:
                numeric_cols = self.data.select_dtypes(include=np.number).columns
                if len(numeric_cols) == 0:
                    print("No numeric columns available.")
                else:
                    self.data[numeric_cols] = self.data[numeric_cols].fillna(
                        self.data[numeric_cols].mean()
                    )
                    print("Numeric missing values filled with mean.")
            elif choice == 3:
                value = input("Enter replacement value: ")
                self.data = self.data.fillna(value)
                print("Missing values replaced.")
            elif choice == 4:
                before = len(self.data)
                self.data = self.data.dropna().reset_index(drop=True)
                print("Rows removed:", before - len(self.data))
            elif choice == 5:
                before = len(self.data)
                self.data = self.data.drop_duplicates().reset_index(drop=True)
                print("Duplicate rows removed:", before - len(self.data))
            else:
                print("Invalid choice!")
        except (ValueError, TypeError) as error:
            print("Cleaning error:", error)
    def mathematical_operations(self):
        if not self.check_data():
            return
        print("\n===== MATHEMATICAL OPERATIONS =====")
        print("1. Add a value to a numeric column")
        print("2. Multiply a numeric column")
        print("3. Calculate profit margin (%)")
        print("4. Calculate Sales + Profit")
        try:
            choice = int(input("Enter choice: "))
            if choice in (1, 2):
                col = self.select_column(numeric_only=True)
                if col is None:
                    return
                value = float(input("Enter number: "))
                self.data[col] = self.data[col] + value if choice == 1 else self.data[col] * value
                print("Operation completed.")
                print(self.data[[col]].head())
            elif choice == 3:
                if not {"Sales", "Profit"}.issubset(self.data.columns):
                    print("Sales and Profit columns are required.")
                    return
                self.data["Profit_Margin"] = np.where(
                    self.data["Sales"] != 0,
                    self.data["Profit"] / self.data["Sales"] * 100, np.nan
                )
                print(self.data[["Sales", "Profit", "Profit_Margin"]].head())
            elif choice == 4:
                if not {"Sales", "Profit"}.issubset(self.data.columns):
                    print("Sales and Profit columns are required.")
                    return
                self.data["Sales_Plus_Profit"] = self.data["Sales"] + self.data["Profit"]
                print(self.data[["Sales", "Profit", "Sales_Plus_Profit"]].head())
            else:
                print("Invalid choice!")
        except ValueError:
            print("Enter a valid number.")

    def combine_data(self):
        if not self.check_data():
            return
        print("\n===== COMBINE DATA =====")
        print("1. Concatenate rows from another CSV\n2. Merge using a common column")
        try:
            choice = int(input("Enter choice: "))
            path = input("Enter second CSV file path: ").strip()
            other = pd.read_csv(path)
            if choice == 1:
                self.data = pd.concat([self.data, other], ignore_index=True)
                print("DataFrames concatenated successfully.")
            elif choice == 2:
                print("Current columns:", self.data.columns.tolist())
                print("Other columns:", other.columns.tolist())
                key = input("Enter common column name: ").strip()
                if key not in self.data.columns or key not in other.columns:
                    print("Common column not found.")
                    return
                self.data = pd.merge(
                    self.data, other, on=key, how="inner",
                    suffixes=("_left", "_right")
                )
                print("DataFrames merged successfully.")
            else:
                print("Invalid choice!")
                return
            print(self.data.head())
        except (OSError, pd.errors.ParserError, ValueError) as error:
            print("Combine error:", error)
    def split_data(self):
        if not self.check_data():
            return
        col = self.select_column()
        if col is None:
            return
        print("Available values:", self.data[col].dropna().unique())
        value = input("Enter value to split by: ").strip()
        subset = self.data[self.data[col].astype(str) == value]
        if subset.empty:
            print("No matching rows found.")
            return
        safe_value = "".join(c if c.isalnum() or c in "-_" else "_" for c in value)
        filename = f"split_{col}_{safe_value}.csv"
        subset.to_csv(filename, index=False)
        print(subset)
        print("Saved to:", filename)
    def search_sort_filter(self):
        if not self.check_data():
            return
        print("\n===== SEARCH, SORT AND FILTER =====")
        print("1. Search for a value\n2. Sort by a column")
        print("3. Filter numeric values\n4. Filter by category")
        print("5. Filter by date range")
        try:
            choice = int(input("Enter choice: "))
            if choice == 1:
                col = self.select_column()
                if col is None:
                    return
                value = input("Enter search value: ")
                result = self.data[self.data[col].astype(str).str.contains(
                    value, case=False, na=False
                )]
                print(result if not result.empty else "No matches found.")
            elif choice == 2:
                col = self.select_column(numeric_only=True)
                if col is None:
                    return
                order = input("Ascending? (yes/no): ").strip().lower()
                print(self.data.sort_values(by=col, ascending=(order == "yes")))
            elif choice == 3:
                col = self.select_column(numeric_only=True)
                if col is None:
                    return
                threshold = float(input("Enter minimum value: "))
                print(self.data[self.data[col] >= threshold])
            elif choice == 4:
                col = input("Enter category column (e.g. Region/Product): ").strip()
                if col not in self.data.columns:
                    print("Column not found.")
                    return
                value = input("Enter category value: ")
                print(self.data[self.data[col].astype(str).str.lower() == value.lower()])
            elif choice == 5:
                if "Date" not in self.data.columns:
                    print("Date column not found.")
                    return
                start = pd.to_datetime(input("Start date (YYYY-MM-DD): "), errors="coerce")
                end = pd.to_datetime(input("End date (YYYY-MM-DD): "), errors="coerce")
                if pd.isna(start) or pd.isna(end) or start > end:
                    print("Invalid date range.")
                    return
                dates = pd.to_datetime(self.data["Date"], errors="coerce")
                print(self.data[dates.between(start, end)])
            else:
                print("Invalid choice!")
        except ValueError:
            print("Enter a valid value.")

    def aggregate_functions(self):
        if not self.check_data():
            return
        print("\n===== AGGREGATE FUNCTIONS =====")
        print("1. Sum\n2. Mean\n3. Count\n4. Group by Region")
        print("5. Group by Product\n6. Group and transform")
        try:
            choice = int(input("Enter choice: "))
            numeric = self.data.select_dtypes(include=np.number).columns.tolist()
            if choice in (1, 2, 3):
                col = self.select_column(numeric_only=True)
                if col is None:
                    return
                if choice == 1:
                    print("Sum:", self.data[col].sum())
                elif choice == 2:
                    print("Mean:", self.data[col].mean())
                else:
                    print("Count:", self.data[col].count())
            elif choice in (4, 5):
                col = "Region" if choice == 4 else "Product"
                if col not in self.data.columns:
                    print(f"{col} column not found.")
                    return
                if not numeric:
                    print("No numeric columns available.")
                    return
                print(self.data.groupby(col)[numeric].sum())
            elif choice == 6:
                if not {"Region", "Sales"}.issubset(self.data.columns):
                    print("Region and Sales columns required.")
                    return
                self.data["Region_Average_Sales"] = self.data.groupby("Region")["Sales"].transform("mean")
                print(self.data[["Region", "Sales", "Region_Average_Sales"]].head())
            else:
                print("Invalid choice!")
        except (ValueError, KeyError) as error:
            print("Aggregation error:", error)
    def statistical_analysis(self):
        if not self.check_data():
            return
        numeric = self.data.select_dtypes(include=np.number)
        if numeric.empty:
            print("No numeric columns available.")
            return
        print("\n===== STATISTICAL ANALYSIS =====")
        print("1. Descriptive statistics\n2. Standard deviation")
        print("3. Variance\n4. Percentile/quantile\n5. Correlation matrix")
        try:
            choice = int(input("Enter choice: "))
            if choice == 1:
                print(numeric.describe())
            elif choice == 2:
                print(numeric.std())
            elif choice == 3:
                print(numeric.var())
            elif choice == 4:
                q = float(input("Enter percentile (0-100): "))
                if 0 <= q <= 100:
                    print(numeric.quantile(q / 100))
                else:
                    print("Percentile must be between 0 and 100.")
            elif choice == 5:
                print(numeric.corr())
            else:
                print("Invalid choice!")
        except ValueError:
            print("Enter a valid number.")

    def create_pivot_table(self):
        if not self.check_data():
            return
        print("\n===== CREATE PIVOT TABLE =====")
        print("Columns:", self.data.columns.tolist())
        index_col = input("Enter row grouping column: ").strip()
        value_col = input("Enter numeric value column: ").strip()
        if index_col not in self.data.columns or value_col not in self.data.columns:
            print("Invalid column name.")
            return
        if not pd.api.types.is_numeric_dtype(self.data[value_col]):
            print("Value column must be numeric.")
            return
        columns_col = input("Column grouping (or Enter to skip): ").strip()
        try:
            if columns_col:
                if columns_col not in self.data.columns:
                    print("Invalid column grouping.")
                    return
                pivot = pd.pivot_table(
                    self.data, index=index_col, columns=columns_col,
                    values=value_col, aggfunc="sum", fill_value=0
                )
            else:
                pivot = pd.pivot_table(
                    self.data, index=index_col, values=value_col,
                    aggfunc=["sum", "mean", "count"]
                )
            print(pivot)
            if input("Save pivot table to CSV? (yes/no): ").lower() == "yes":
                pivot.to_csv("sales_pivot_table.csv")
                print("Saved to sales_pivot_table.csv")
        except (ValueError, KeyError) as error:
            print("Pivot table error:", error)

    def numpy_operations(self):
        if not self.check_data():
            return
        numeric = self.data.select_dtypes(include=np.number)
        if numeric.empty:
            print("No numeric columns available.")
            return
        array = numeric.to_numpy()
        print("\n===== NUMPY ARRAY OPERATIONS =====")
        print("1. Display array\n2. Indexing\n3. Slicing")
        print("4. Element-wise addition\n5. Element-wise multiplication")
        try:
            choice = int(input("Enter choice: "))
            if choice == 1:
                print(array)
            elif choice == 2:
                row = int(input("Enter row index: "))
                col = int(input("Enter column index: "))
                print("Element:", array[row, col])
            elif choice == 3:
                start = int(input("Starting row: "))
                end = int(input("Ending row: "))
                print(array[start:end, :])
            elif choice == 4:
                print(array + float(input("Number to add: ")))
            elif choice == 5:
                print(array * float(input("Number to multiply: ")))
            else:
                print("Invalid choice!")
        except (ValueError, IndexError) as error:
            print("Array operation error:", error)
    def visualize_data(self):
        if not self.check_data():
            return
        print("\n===== DATA VISUALIZATION =====")
        print("1. Bar chart\n2. Line chart\n3. Scatter plot")
        print("4. Pie chart\n5. Histogram\n6. Stack plot")
        print("7. Subplots\n8. Seaborn heatmap\n9. Seaborn box plot")
        fig = None
        try:
            choice = int(input("Enter choice: "))
            sns.set_theme(style="whitegrid")
            fig, ax = plt.subplots(figsize=(9, 5))

            if choice == 1:
                if not {"Product", "Sales"}.issubset(self.data.columns):
                    print("Product and Sales columns required.")
                    plt.close(fig)
                    return
                self.data.groupby("Product")["Sales"].sum().plot(kind="bar", ax=ax)
                ax.set(title="Total Sales by Product", xlabel="Product", ylabel="Sales")
            elif choice == 2:
                if not {"Date", "Sales"}.issubset(self.data.columns):
                    print("Date and Sales columns required.")
                    plt.close(fig)
                    return
                temp = self.data.copy()
                temp["Date"] = pd.to_datetime(temp["Date"], errors="coerce")
                temp = temp.dropna(subset=["Date"])
                temp.groupby("Date")["Sales"].sum().sort_index().plot(
                    kind="line", marker="o", ax=ax
                )
                ax.set(title="Sales Trend Over Time", xlabel="Date", ylabel="Sales")
            elif choice == 3:
                if not {"Sales", "Profit"}.issubset(self.data.columns):
                    print("Sales and Profit columns required.")
                    plt.close(fig)
                    return
                sns.scatterplot(data=self.data, x="Sales", y="Profit", ax=ax)
                ax.set_title("Sales vs Profit")
            elif choice == 4:
                if not {"Product", "Sales"}.issubset(self.data.columns):
                    print("Product and Sales columns required.")
                    plt.close(fig)
                    return
                summary = self.data.groupby("Product")["Sales"].sum()
                summary = summary[summary > 0]
                if summary.empty:
                    print("No positive sales values to display.")
                    plt.close(fig)
                    return
                plt.close(fig)
                fig, ax = plt.subplots(figsize=(7, 7))
                summary.plot(kind="pie", autopct="%1.1f%%", ax=ax)
                ax.set_ylabel("")
                ax.set_title("Sales Share by Product")
            elif choice == 5:
                col = self.select_column(numeric_only=True)
                if col is None:
                    plt.close(fig)
                    return
                sns.histplot(data=self.data, x=col, kde=True, ax=ax)
                ax.set_title(f"Distribution of {col}")
            elif choice == 6:
                required = {"Region", "Product", "Sales"}
                if not required.issubset(self.data.columns):
                    print("Region, Product and Sales columns required.")
                    plt.close(fig)
                    return
                pivot = self.data.pivot_table(
                    index="Product", columns="Region", values="Sales",
                    aggfunc="sum", fill_value=0
                )
                plt.close(fig)
                fig, ax = plt.subplots(figsize=(9, 5))
                x = np.arange(len(pivot.index))
                bottom = np.zeros(len(pivot.index))
                for region in pivot.columns:
                    vals = pivot[region].to_numpy()
                    ax.bar(x, vals, bottom=bottom, label=str(region))
                    bottom += vals
                ax.set_xticks(x)
                ax.set_xticklabels(pivot.index, rotation=30)
                ax.set_title("Stacked Sales by Product and Region")
                ax.set_ylabel("Sales")
                ax.legend(title="Region")
            elif choice == 7:
                required = {"Product", "Region", "Sales", "Profit"}
                if not required.issubset(self.data.columns):
                    print("Product, Region, Sales and Profit columns required.")
                    plt.close(fig)
                    return
                plt.close(fig)
                fig, axes = plt.subplots(2, 2, figsize=(12, 8))
                self.data.groupby("Product")["Sales"].sum().plot(
                    kind="bar", ax=axes[0, 0], title="Sales by Product"
                )
                self.data.groupby("Region")["Profit"].sum().plot(
                    kind="bar", ax=axes[0, 1], title="Profit by Region"
                )
                sns.scatterplot(data=self.data, x="Sales", y="Profit", ax=axes[1, 0])
                axes[1, 0].set_title("Sales vs Profit")
                sns.histplot(data=self.data, x="Sales", ax=axes[1, 1])
                axes[1, 1].set_title("Sales Distribution")
                fig.tight_layout()
            elif choice == 8:
                numeric = self.data.select_dtypes(include=np.number)
                if numeric.shape[1] < 2:
                    print("At least two numeric columns required.")
                    plt.close(fig)
                    return
                sns.heatmap(numeric.corr(), annot=True, cmap="coolwarm", ax=ax)
                ax.set_title("Correlation Heatmap")
            elif choice == 9:
                col = self.select_column(numeric_only=True)
                if col is None:
                    plt.close(fig)
                    return
                sns.boxplot(data=self.data, y=col, ax=ax)
                ax.set_title(f"Box Plot of {col}")
            else:
                print("Invalid choice!")
                plt.close(fig)
                return
            fig.tight_layout()
            self.last_plot = fig
            plt.show()
        except (ValueError, KeyError, TypeError) as error:
            if fig is not None:
                plt.close(fig)
            print("Visualization error:", error)
    def save_visualization(self):
        if self.last_plot is None:
            print("Create a visualization first!")
            return
        filename = input("Filename (e.g. sales_chart.png): ").strip()
        if not filename:
            print("Filename cannot be empty.")
            return
        if not filename.lower().endswith((".png", ".jpg", ".jpeg")):
            print("Use a .png, .jpg or .jpeg filename.")
            return
        try:
            self.last_plot.savefig(filename, dpi=300, bbox_inches="tight")
            print("Visualization saved:", filename)
        except (OSError, ValueError) as error:
            print("Save error:", error)
    def export_data(self):
        if not self.check_data():
            return
        filename = input("Output CSV filename: ").strip()
        if not filename:
            print("Filename cannot be empty.")
            return
        if not filename.lower().endswith(".csv"):
            filename += ".csv"
        try:
            self.data.to_csv(filename, index=False)
            print("Analysis data exported to:", filename)
        except OSError as error:
            print("Export error:", error)
    def main(self):
        while True:
            print("\n" + "=" * 55)
            print("       PANDAS ANALYZER & DATA VISUALIZATION")
            print("=" * 55)
            print("1.  Load Dataset")
            print("2.  Explore Data")
            print("3.  Clean Missing Data")
            print("4.  Mathematical Operations")
            print("5.  Combine DataFrames")
            print("6.  Split DataFrame")
            print("7.  Search, Sort and Filter")
            print("8.  Aggregate Functions")
            print("9.  Statistical Analysis")
            print("10. Create Pivot Table")
            print("11. NumPy Array Operations")
            print("12. Data Visualization")
            print("13. Save Visualization")
            print("14. Export Analysis Data")
            print("15. Exit")
            print("=" * 55)
            try:
                choice = int(input("Enter your choice: "))
                actions = {
                    1: lambda: self.load_data(input("Enter CSV file path: ").strip()),
                    2: self.explore_data,
                    3: self.clean_data,
                    4: self.mathematical_operations,
                    5: self.combine_data,
                    6: self.split_data,
                    7: self.search_sort_filter,
                    8: self.aggregate_functions,
                    9: self.statistical_analysis,
                    10: self.create_pivot_table,
                    11: self.numpy_operations,
                    12: self.visualize_data,
                    13: self.save_visualization,
                    14: self.export_data,
                }
                if choice == 15:
                    print("Thank you for using Pandas Analyzer!")
                    break
                elif choice in actions:
                    actions[choice]()
                else:
                    print("Invalid choice! Select from 1 to 15.")
            except ValueError:
                print("Please enter a valid menu number.")


In [ ]:
analyzer = SalesDataAnalyzer("sales_data.csv")
analyzer.main()
